# 18-delta-lake

18-delta-lake/01-delta-lake-patterns.py

Delta Lake: ACID writes, UPDATE, DELETE, MERGE (upsert), time travel,
VACUUM, schema enforcement, and schema evolution.

Prerequisites:
    pip install delta-spark

Version matrix (adjust spark_session.py if needed):
    PySpark 3.3 → delta-spark_2.12:2.3.0
    PySpark 3.4 → delta-spark_2.12:2.4.0
    PySpark 3.5 → delta-spark_2.12:3.2.0

Run:
    python 01-delta-lake-patterns.py
Delta log: data/output/delta/employees/_delta_log/

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# ── SparkSession (delta=True adds Delta jars + SQL extensions) ────────────────
spark = get_spark("18-delta-lake", delta=True)
dfs   = register_views(spark)
emp   = dfs["employees"]
sal   = dfs["salary_history"]
from delta import DeltaTable


# ── Section 1: Write a Delta table ───────────────────────────────────────────
# format("delta") writes Parquet data files + a _delta_log/ transaction log.
# The log is what gives Delta ACID guarantees (atomicity, isolation, etc.).
print("\n" + "="*55)
print("  Section 1: Write a Delta table")
print("="*55)

delta_emp_path = output_path("delta/employees")
emp.write.format("delta").mode("overwrite").save(delta_emp_path)
print("Delta employees written to:", delta_emp_path)
# Check data/output/delta/employees/_delta_log/00000000000000000000.json
# to see the first commit entry


# ── Section 2: Read a Delta table ────────────────────────────────────────────
# Reading is identical to Parquet, just format("delta").
# Delta always returns a consistent snapshot at the latest version.
print("\n" + "="*55)
print("  Section 2: Read a Delta table")
print("="*55)

df_delta = spark.read.format("delta").load(delta_emp_path)
df_delta.printSchema()
df_delta.show(3)
print("Total rows:", df_delta.count())   # 41 rows (all employees including emp 35 Terminated)


# ── Section 3: Delta ACID UPDATE ─────────────────────────────────────────────
# UPDATE modifies rows matching a condition in-place (creates a new version).
# Flaw in data: emp 22 has email=None — we fix it here.
print("\n" + "="*55)
print("  Section 3: ACID UPDATE — fix emp 22 NULL email")
print("="*55)

dt = DeltaTable.forPath(spark, delta_emp_path)
dt.update(
    condition=F.col("emp_id") == 22,
    set={"email": F.lit("v.lee@corp.com")}
)
spark.read.format("delta").load(delta_emp_path) \
     .filter(F.col("emp_id") == 22) \
     .select("emp_id", "first_name", "email") \
     .show()
# Expected: email = v.lee@corp.com (was None)
# Delta log: version 1 entry added to _delta_log/


# ── Section 4: Delta ACID DELETE ─────────────────────────────────────────────
# DELETE removes matching rows and records the operation in the log.
# Old data files are retained until VACUUM runs (time travel still works).
print("\n" + "="*55)
print("  Section 4: ACID DELETE — remove terminated employee (emp 35)")
print("="*55)

dt.delete(F.col("emp_id") == 35)
after_del = spark.read.format("delta").load(delta_emp_path).count()
print(f"After delete: {after_del} rows (was 41, now 40)")
# Delta log: version 2 entry; old file with emp 35 still on disk until VACUUM


# ── Section 5: Time travel — query by version ─────────────────────────────────
# Delta retains all historical data files until VACUUM.
# versionAsOf lets you read any past snapshot by commit number.
#   version 0 = original write  (41 rows, emp 22 email=None, emp 35 present)
#   version 1 = after UPDATE    (41 rows, emp 22 email fixed)
#   version 2 = after DELETE    (40 rows, emp 35 removed)
print("\n" + "="*55)
print("  Section 5: Time travel — versionAsOf")
print("="*55)

v0 = spark.read.format("delta").option("versionAsOf", 0).load(delta_emp_path)
v2 = spark.read.format("delta").option("versionAsOf", 2).load(delta_emp_path)
print(f"Version 0 count: {v0.count()} (original write — 41 rows)")
print(f"Version 2 count: {v2.count()} (after DELETE — 40 rows)")

# Confirm version 0 still has emp 22 with NULL email
v0.filter(F.col("emp_id") == 22).select("emp_id", "first_name", "email").show()

# Confirm version 0 still has the terminated employee
v0.filter(F.col("emp_id") == 35).select("emp_id", "first_name", "status").show()


# ── Section 6: Time travel — query by timestamp ──────────────────────────────
# timestampAsOf reads the latest version that existed at or before the given ts.
# Useful for auditing ("what did this table look like last Tuesday?").
print("\n" + "="*55)
print("  Section 6: Time travel — timestampAsOf")
print("="*55)

from datetime import datetime
ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
current_via_ts = spark.read.format("delta").option("timestampAsOf", ts).load(delta_emp_path)
print(f"Rows as of {ts}: {current_via_ts.count()}")
# Should equal version 2 count (40) — the most recent committed version


# ── Section 7: MERGE (upsert) ─────────────────────────────────────────────────
# MERGE is Delta's most powerful operation:
#   - whenMatchedUpdateAll  → UPDATE existing rows where keys match
#   - whenNotMatchedInsertAll → INSERT rows from source that are new
# One atomic operation, no double-write risk.
print("\n" + "="*55)
print("  Section 7: MERGE — upsert (update emp 19 + insert emp 42)")
print("="*55)

# emp 19 has salary=0.0 (data entry flaw); we correct it to 115000.0
# emp 42 is a brand-new employee not in the table
staging = spark.createDataFrame(
    [
        (19, "Samuel", "Clark",  "s.clark@corp.com",  "212-555-0119", 2, 3,    "Regional Sales Lead", "2017-08-14", 115000.0, "Active", None),
        (42, "Zoe",    "Murphy", "z.murphy@corp.com", "415-555-0142", 1, 2,    "ML Engineer",         "2024-06-01", 135000.0, "Active", None),
    ],
    schema=(
        "emp_id INT, first_name STRING, last_name STRING, email STRING, "
        "phone STRING, dept_id INT, manager_id INT, job_title STRING, "
        "hire_date STRING, salary DOUBLE, status STRING, termination_date STRING"
    )
)
staging = (
    staging
    .withColumn("hire_date",        F.to_date("hire_date"))
    .withColumn("termination_date", F.to_date("termination_date"))
)

dt.alias("target").merge(
    staging.alias("source"),
    "target.emp_id = source.emp_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

merged = spark.read.format("delta").load(delta_emp_path)
print(f"After merge: {merged.count()} rows (was 40; +1 new emp)")
merged.filter(F.col("emp_id").isin(19, 42)).select("emp_id", "first_name", "salary").show()
# emp 19: salary now 115000.0 (corrected); emp 42: new row inserted


# ── Section 8: Delta table history ───────────────────────────────────────────
# dt.history() returns the full transaction log as a DataFrame.
# Each row is one commit: WRITE(v0), UPDATE(v1), DELETE(v2), MERGE(v3).
# Inspect operationMetrics to see rows inserted/updated/deleted per operation.
print("\n" + "="*55)
print("  Section 8: Delta table history (transaction log)")
print("="*55)

dt.history().select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)
# Spark UI → Storage tab won't show Delta history; use the output above
# or inspect _delta_log/*.json files directly


# ── Section 9: VACUUM (remove old file versions) ─────────────────────────────
# Default retention is 7 days; VACUUM deletes files older than the threshold.
# After VACUUM, time travel to pruned versions is no longer possible.
# DO NOT use retentionHours=0 in production — it breaks concurrent readers.
print("\n" + "="*55)
print("  Section 9: VACUUM — purge old data files")
print("="*55)

spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
dt.vacuum(retentionHours=0)
print("Vacuum complete — files older than 0h removed")
print("Time travel to version 0 will now raise AnalysisException")


# ── Section 10: Schema enforcement and schema evolution ───────────────────────
# Enforcement (default ON): Delta rejects writes whose schema doesn't match
# the table's registered schema — prevents silent data corruption.
#
# Evolution (opt-in): pass mergeSchema=true to allow adding new columns.
# Existing rows get NULL for the new column.
print("\n" + "="*55)
print("  Section 10: Schema enforcement + schema evolution")
print("="*55)

# Schema enforcement — wrong columns → blocked
try:
    spark.createDataFrame(
        [(99, "Wrong", "Schema", 12345)],
        ["emp_id", "a", "b", "c"]
    ).write.format("delta").mode("append").save(delta_emp_path)
except Exception as e:
    print(f"Schema enforcement blocked write: {type(e).__name__}")
    # Expected: AnalysisException — schema mismatch

# Schema evolution — add 'bonus' column via mergeSchema
emp_with_bonus = emp.withColumn("bonus", F.col("salary") * 0.1)
emp_with_bonus.write.format("delta").mode("append") \
    .option("mergeSchema", "true") \
    .save(delta_emp_path)

evolved_schema = spark.read.format("delta").load(delta_emp_path)
print("Schema after evolution:")
evolved_schema.printSchema()   # 'bonus' column now present
evolved_schema.filter(F.col("bonus").isNotNull()).select("emp_id", "salary", "bonus").show(3)
# Rows from earlier writes have bonus=NULL; new rows have bonus=salary*0.1


# ── Done ──────────────────────────────────────────────────────────────────────
stop_and_wait(spark)